# AI-Based Skin Disease Detection: Deep Learning Model Training

This notebook trains a deep Convolutional Neural Network (CNN) using **Transfer Learning** (MobileNetV2 backbone) to detect common skin diseases, including:
1. **Acne**
2. **Eczema**
3. **Psoriasis**
4. **Ringworm**
5. **Vitiligo**
6. **Melanoma**
7. **Melanocytic Nevus**
8. **Dermatitis**

This notebook can be run locally or uploaded directly to **Google Colab** / **Kaggle** with a free GPU (T4/P100) for rapid training.

In [ ]:
# 1. Environment Setup & Dependency Imports
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import MobileNetV2
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, ReduceLROnPlateau
from sklearn.metrics import classification_report, confusion_matrix

print(f"TensorFlow Version: {tf.__version__}")
print(f"GPU Available: {len(tf.config.list_physical_devices('GPU')) > 0}")

### 2. Configuration & Hyperparameters
We define our input image dimensions, batch size, target classes, and training epochs.

In [ ]:
IMG_HEIGHT = 224
IMG_WIDTH = 224
BATCH_SIZE = 32
EPOCHS = 25
LEARNING_RATE = 1e-4

CLASS_NAMES = [
    "acne",
    "eczema",
    "psoriasis",
    "ringworm",
    "vitiligo",
    "melanoma",
    "melanocytic_nevus",
    "dermatitis"
]
NUM_CLASSES = len(CLASS_NAMES)
print(f"Configured {NUM_CLASSES} classes: {CLASS_NAMES}")

### 3. Dataset Ingestion & Pre-Splitting
Following **ML Best Practices**, we strictly partition our data into **Training (70%)**, **Validation (15%)**, and **Test (15%)** sets **prior** to running data augmentation pipelines to prevent data leakage.

In [ ]:
DATA_DIR = "./dataset"  # Path containing subfolders per class

# Check if dataset exists locally, otherwise simulate or prepare directory
if not os.path.exists(DATA_DIR):
    print(f"Notice: '{DATA_DIR}' directory not found. Please link or extract your images into {DATA_DIR}/<class_name>/")
    print("Example: dataset/acne/, dataset/eczema/, etc.")
else:
    # Load train and validation splits
    train_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=0.3,
        subset="training",
        seed=42,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    # Split the remaining 30% equally into validation (15%) and test (15%)
    val_test_ds = tf.keras.utils.image_dataset_from_directory(
        DATA_DIR,
        validation_split=0.3,
        subset="validation",
        seed=42,
        image_size=(IMG_HEIGHT, IMG_WIDTH),
        batch_size=BATCH_SIZE,
        label_mode="categorical"
    )

    val_batches = tf.data.experimental.cardinality(val_test_ds) // 2
    val_ds = val_test_ds.take(val_batches)
    test_ds = val_test_ds.skip(val_batches)

    # Optimize I/O pipeline
    AUTOTUNE = tf.data.AUTOTUNE
    train_ds = train_ds.cache().shuffle(1000).prefetch(buffer_size=AUTOTUNE)
    val_ds = val_ds.cache().prefetch(buffer_size=AUTOTUNE)
    test_ds = test_ds.cache().prefetch(buffer_size=AUTOTUNE)
    print("Datasets successfully initialized and partitioned.")

### 4. Data Augmentation & Transfer Learning Architecture
Skin lesion datasets often suffer from class imbalances and orientation variance. We introduce real-time data augmentations (random rotations, zoom, horizontal and vertical flips) within the model architecture itself, coupled with the pre-trained **MobileNetV2** backbone initialized with ImageNet weights.

In [ ]:
def build_skin_disease_model(num_classes=NUM_CLASSES):
    # Data Augmentation pipeline
    data_augmentation = keras.Sequential([
        layers.RandomFlip("horizontal_and_vertical"),
        layers.RandomRotation(0.2),
        layers.RandomZoom(0.15),
        layers.RandomContrast(0.1)
    ], name="data_augmentation")

    # Pretrained MobileNetV2 base (weights='imagenet', include_top=False)
    base_model = MobileNetV2(
        input_shape=(IMG_HEIGHT, IMG_WIDTH, 3),
        include_top=False,
        weights='imagenet'
    )
    base_model.trainable = False  # Freeze base during initial phase

    # Construct full model
    inputs = keras.Input(shape=(IMG_HEIGHT, IMG_WIDTH, 3))
    x = data_augmentation(inputs)
    x = layers.Rescaling(1./127.5, offset=-1)(x)  # Normalization for MobileNetV2
    x = base_model(x, training=False)
    x = layers.GlobalAveragePooling2D()(x)
    x = layers.BatchNormalization()(x)
    x = layers.Dropout(0.35)(x)
    x = layers.Dense(256, activation='relu')(x)
    x = layers.Dropout(0.2)(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='prediction_output')(x)

    model = keras.Model(inputs, outputs, name="SkinDisease_MobileNetV2")
    return model

model = build_skin_disease_model()
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=LEARNING_RATE),
    loss='categorical_crossentropy',
    metrics=['accuracy', keras.metrics.AUC(name='auc'), keras.metrics.Precision(name='precision'), keras.metrics.Recall(name='recall')]
)
model.summary()

### 5. Training Callbacks & Model Fitting
We configure:
- `EarlyStopping` to avoid overfitting on the training data.
- `ReduceLROnPlateau` to decrease the learning rate when validation loss plateaus.
- `ModelCheckpoint` to save the best model weights.

In [ ]:
callbacks = [
    EarlyStopping(monitor='val_loss', patience=6, restore_best_weights=True, verbose=1),
    ReduceLROnPlateau(monitor='val_loss', factor=0.2, patience=3, min_lr=1e-6, verbose=1),
    ModelCheckpoint(
        filepath='../models/skin_disease_model.keras',
        monitor='val_accuracy',
        save_best_only=True,
        verbose=1
    )
]

# Note: Run model.fit when dataset is extracted into DATA_DIR
# history = model.fit(
#     train_ds,
#     validation_data=val_ds,
#     epochs=EPOCHS,
#     callbacks=callbacks
# )
print("Training configuration and callbacks ready.")

### 6. Fine-Tuning the Base Model
After initial convergence of the top dense layers, unfreeze the top layers of the base MobileNetV2 network and fine-tune with a reduced learning rate.

In [ ]:
# Unfreeze base model for fine-tuning
# base_model = model.get_layer('mobilenetv2_1.00_224')
# base_model.trainable = True
# for layer in base_model.layers[:100]:
#     layer.trainable = False  # Keep early feature extractors frozen

# model.compile(
#     optimizer=keras.optimizers.Adam(learning_rate=1e-5),
#     loss='categorical_crossentropy',
#     metrics=['accuracy', keras.metrics.AUC(name='auc')]
# )
print("Fine-tuning recipe defined.")

### 7. Evaluation: Confusion Matrix & Classification Metrics
Once training completes, assess multi-class performance on the held-out **Test Set**.

In [ ]:
# Code template for test evaluation:
# test_loss, test_acc, test_auc, test_prec, test_rec = model.evaluate(test_ds)
# print(f"Test Accuracy: {test_acc*100:.2f}%")
# print(f"Test AUC: {test_auc:.4f}")

# Generate Confusion Matrix:
# y_pred_probs = model.predict(test_ds)
# y_pred = np.argmax(y_pred_probs, axis=1)
# y_true = np.concatenate([y for x, y in test_ds], axis=0).argmax(axis=1)
# cm = confusion_matrix(y_true, y_pred)

# plt.figure(figsize=(10, 8))
# sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
# plt.title('Skin Disease Confusion Matrix')
# plt.xlabel('Predicted Label')
# plt.ylabel('Ground Truth')
# plt.show()
print("Evaluation scripts ready.")

### 8. Exporting the Model for the Flask Web Application
Save the finalized model into the `models/` directory for immediate inference by the web app.

In [ ]:
output_model_path = os.path.join("..", "models", "skin_disease_model.keras")
model.save(output_model_path)
print(f"Model saved to: {output_model_path}")